# Load input data

In [1]:
from functions import get_experiment_data

X_df4, meta_df4, batch_df4, _ = get_experiment_data(
    design_id="df4" ,
    file_path="/DATA/WGS_study/YSL/projects/Data",
    verbose=True,
)

Design ID               : df4
Design name             : prep_genus_count
Description             : Preprocessed HIVRC, genus-level, raw count
Aggregation             : genus
Normalize               : False
Cleanset Filtering      : True
Subset studies          : None
OTU zero-prev           : None
Sample zero-prev        : None
----------------------------------------------------------------------
feature_table           : (936, 668)
meta_data               : (936, 11)
n_batches               : 14


# Trial investigation

In [2]:
import pandas as pd
import glob

# df4 CSV 파일 찾기
df4_p1_res = pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df4/phase1/optuna_trials_2026-04-24.csv")
print(f"전체 trial 수: {len(df4_p1_res)}")

# 필터링
df_clean = df4_p1_res[
    (df4_p1_res['permanova'] > 0.01) &
    (df4_p1_res['permanova'] < 0.1) &
    (df4_p1_res['noise_ratio'] < 0.95) &
    (df4_p1_res['n_clusters'] >= 5)
].sort_values(['noise_ratio', 'permanova', 'n_clusters'], ascending=[True, True, False])

print(f"의미있는 trial 수: {len(df_clean)}")
print(df_clean[['trial_number', 'permanova', 'n_clusters', 'noise_ratio', 'cutoff']].head(20))

전체 trial 수: 3081
의미있는 trial 수: 5
      trial_number  permanova  n_clusters  noise_ratio  cutoff
2320          3927   0.046420           5     0.227564     0.1
2901          4746   0.030847           5     0.471154     0.1
2219          3784   0.046054           5     0.719017     0.1
2859          4684   0.058532           5     0.721154     0.1
2203          3764   0.071533           5     0.838675     0.1


# Best trial selection (trial 4746)

In [3]:
print(df4_p1_res[df4_p1_res['trial_number'] == 4746].T)

                              2901
cutoff                         0.1
trial_number                  4746
base_dim                       512
n_layers                         2
latent_dim                      50
activation              leaky_relu
strategy                     halve
dropout_rate                   0.5
epochs                         150
learning_rate             0.000014
loss                    430.908813
permanova                 0.030847
n_clusters                       5
noise_ratio               0.471154
min_cluster_size                 5
min_samples_token              5.0
batch_size                     128
init               kaiming_uniform
beta_kl                   0.215791
weight_decay                   0.0
grad_clip_norm            1.070277
csm                            eom
kl_warmup_ratio           0.742873
norm                     layernorm
scheduler                   cosine


# Run phase 2

In [4]:
from prototype_updated_phase2 import main
main(
    experiment_name="df4",
    phase=2,
    best_trial_number=4746,
    trial_res_file_phase2="/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df4/phase1/optuna_trials_2026-04-24.csv"
)

2026-05-08 16:08:26 | 로그 저장 경로: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df4/phase2/latentgee_prototype_cutoff_df4_pid1073392_2026-05-08.log
2026-05-08 16:08:26 | LatentGEE 시작 | experiment=df4 | phase=2
2026-05-08 16:08:26 | config snapshot saved: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df4/phase2/config_used.yaml
2026-05-08 16:08:26 | python == 3.10.20
2026-05-08 16:08:26 | torch == 2.2.2
2026-05-08 16:08:26 | numpy == 1.26.4
2026-05-08 16:08:26 | scikit-learn == 1.7.2
2026-05-08 16:08:26 | optuna == 3.6.1
2026-05-08 16:08:26 | hdbscan == 0.8.41
Design ID               : df4
Design name             : prep_genus_count
Description             : Preprocessed HIVRC, genus-level, raw count
Aggregation             : genus
Normalize               : False
Cleanset Filtering      : True
Subset studies          : None
OTU zero-prev           : None
Sample zero-prev        : None
----------------------------------------------------------------------
feature_table        

In [5]:
# from functions import zero_filter
def zero_filter(df, meta, cutoff):
    batch="Study"
    prevalence = (df > 0).sum(axis=0) / df.shape[0]
    df = df.loc[:, prevalence > best_cutoff].copy()

    row_sums = df.sum(axis=1)
    keep_sample = row_sums > 0
    df = df.loc[keep_sample].copy()
    meta = meta.loc[keep_sample.values].reset_index(drop=True)
    
    assert len(df) == len(meta)
    assert all(df.index.astype(str) == meta["SeqID"].astype(str))
    
    return df, meta, meta[batch]

best_cutoff = 0.1
X_df4_filtered, meta_df4_filtered, batch_df4_filtered = zero_filter(X_df4, meta_df4, best_cutoff)

    

In [10]:
X_df4_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df4_filtered_cutoff0.1.csv", index=True)
meta_df4_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/meta_df4_filtered_cutoff0.1.csv", index=True)
batch_df4_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/batch_df4_filtered_cutoff0.1.csv", index=True)

In [7]:
import numpy as np

# X_corrected 로드
X_corrected_df4 = pd.read_csv(
    "/DATA/WGS_study/YSL/projects/latentgee/examples/results/df4/phase2/df4_X_corrected_trial4746_cutoff0.1_2026-04-27.csv",
    index_col=0
)
# inf, NaN 처리
X_corrected_df4_clean = X_corrected_df4.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
row_sums = X_corrected_df4_clean.sum(axis=1).replace(0, 1)
X_corrected_df4_clean = X_corrected_df4_clean.div(row_sums, axis=0)

print(f"shape: {X_corrected_df4_clean.shape}")
print(f"NaN 수: {X_corrected_df4_clean.isna().sum().sum()}")
print(f"inf 수: {np.isinf(X_corrected_df4_clean.values).sum()}")

shape: (936, 135)
NaN 수: 0
inf 수: 0


In [8]:
from functions import evaluate_batch_correction
df4_result = evaluate_batch_correction(
    X_before     = X_df4_filtered,
    X_after      = X_corrected_df4_clean,
    batch_labels = batch_df4_filtered,
    bio_labels   = meta_df4_filtered['hivstatus'],
    renormalize  = False,
    label        = "df4 — LatentGEE"
)



  df4 — LatentGEE
                        Before   After  Change
Metric                                        
PERMANOVA R² (batch) ↓  0.2659  0.0146 -0.2513
PERMANOVA R² (bio) ↑    0.0039  0.0011 -0.0028
kBET acceptance rate ↑  0.0053  0.9071  0.9017
ASW (batch) → 0        -0.1328 -0.0322  0.1006
ASW (bio) ↑            -0.0266 -0.0010  0.0256

